last modified date : 2026.07  
제작 : 모두의연구소

# Day 2 & 3 통합 실습 — Advanced·Modular RAG + RAGAS 평가 + Langfuse 관측

이 통합 실습 노트북은 **Advanced·Modular RAG 파이프라인 구축**, **RAGAS 정량 평가**, 그리고 **Langfuse 기반 LLM Observability(관측성) & 원인 분석**을 한 곳에서 체계적으로 학습할 수 있도록 하나로 통합 정리되었습니다.

---
## 📌 전체 학습 로드맵 (Table of Contents)

- **[Part 1] 통합 설치 및 환경 설정 (Step 0)**
  - Ragas 0.2.10, LangChain 0.2.x, Langfuse 4.x 통합 설치 및 OpenAI / Langfuse API 키 설정
- **[Part 2] KorQuAD v1 기반 Advanced·Modular RAG 구축 (Step 1 ~ Step 5.5)**
  - Naive RAG 베이스라인 → Multi-Query → RAG-Fusion → HyDE → Cross-Encoder Reranker → Advanced RAG 체인 → Self-RAG
- **[Part 3] KorQuAD v1 기반 RAGAS 정량 평가 (Step 6 ~ Step 8)**
  - RAGAS 4대 지표(Faithfulness, Answer Relevance, Context Precision, Context Recall) 계산 및 해석
- **[Part 4] [추가 실습] KLUE-MRC 벤치마크 평가 (Step A ~ Step K)**
  - KLUE-MRC 뉴스 데이터셋 기반 Naive vs Advanced RAG 성능 비교 및 통계적 검정(Paired t-test)
- **[Part 5] Langfuse 기반 LLM Observability & 원인 분석 (Step 9 ~ Step 12)**
  - `@observe` 데코레이터를 활용한 파이프라인 관측(Trace), RAGAS 점수의 Trace 자동 부착(Score Attaching), 대시보드 기반 Root Cause Analysis


# [Part 1] 통합 설치 및 환경 설정

## Step 0 : 패키지 통합 설치와 환경 준비

Advanced RAG, RAGAS 평가, Langfuse 관측에 필요한 패키지들을 버전 충돌 없이 하나의 셀에서 통합 설치합니다.
진행률 출력을 보면서 기다려주세요 (약 3~5분 소요).

In [ ]:
# 1) 기존 langchain / ragas / langfuse 패키지 정리 (버전 충돌 방지)
!pip uninstall -y ragas ragas-experimental langchain langchain-core langchain-community langchain-openai langchain-text-splitters langchain-chroma langfuse

# 2) RAGAS 0.2.10 호환 LangChain 0.2.x 핀 버전 및 Langfuse 4.x 통합 설치
!pip install --no-cache-dir \
    "ragas==0.2.10" \
    "langchain==0.2.17" \
    "langchain-core==0.2.43" \
    "langchain-community==0.2.19" \
    "langchain-openai==0.1.25" \
    "langchain-text-splitters==0.2.4" \
    "langchain-chroma==0.1.4" \
    "langfuse>=4.14,<5" \
    pypdf chromadb tiktoken sentence-transformers datasets nest_asyncio pandas


> ⚠️ **위 설치 셀(Step 0)을 실행한 뒤 반드시 [런타임 > 세션 다시 시작 (Restart session)]을 한 번 눌러주세요.**
>
> 이 셀은 Colab에 기본 설치된 패키지들을 교체하므로 재시작이 필요합니다.
> 재시작 후에는 설치 셀을 다시 실행하지 말고 아래 **키 설정 셀**부터 순서대로 실행하시면 됩니다.

In [ ]:
import os
from google.colab import userdata

# chromadb 익명 통계 전송 끄기
os.environ["ANONYMIZED_TELEMETRY"] = "False"

# 1) OpenAI API Key 설정
if "OPENAI_API_KEY" not in os.environ:
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        print("OpenAI API Key 설정 완료!")
    except Exception as e:
        print("⚠️ OPENAI_API_KEY를 Colab Secrets에서 가져오지 못했습니다:", e)

# 2) Langfuse API Keys & Host 설정 (Part 5 관측용)
try:
    os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY")
    os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY")
    print("Langfuse Keys 설정 완료!")
except Exception as e:
    print("⚠️ Langfuse Keys가 Colab Secrets에 설정되지 않았습니다 (Part 5 실행 전 설정 필요):", e)

# Cloudflare 터널 주소 또는 Localhost URL (본인의 터널 주소로 변경하세요)
LANGFUSE_HOST = "https://prefer-filename-glen-che.trycloudflare.com"
os.environ["LANGFUSE_HOST"] = LANGFUSE_HOST
print("Langfuse Host:", os.environ.get("LANGFUSE_HOST"))


# [Part 2] KorQuAD v1 기반 Advanced·Modular RAG 구축


## Step 1 : KorQuAD v1 위에서 Naive RAG 베이스라인 만들기

Day 1에서 만든 RAG 파이프라인을 한국어 QA 벤치마크 **KorQuAD v1** 위에 다시 한 번 올립니다. 이후 단계는 모두 이 베이스라인 위에 ‘덧붙이는’ 방식입니다.

- HuggingFace `datasets` 로 KorQuAD v1 자동 다운로드 (별도 PDF 업로드 불필요)
- 일부만 샘플링해 토큰 비용 통제 (unique context 약 200개)
- Embedding → VectorStore → Retriever → LLM
- 검색 전략은 단순 `similarity` (top-k)

**📥 데이터셋**: <https://huggingface.co/datasets/KorQuAD/squad_kor_v1>

In [3]:
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import tiktoken, random

tokenizer = tiktoken.get_encoding("cl100k_base")
def tiktoken_len(text):
    return len(tokenizer.encode(text))

# 1) 데이터셋 로드 + 2000개 샘플링 + context 중복 제거 → unique 약 800개
#    (Vector DB 가 크면 Reranker 의 정밀도 개선 효과가 더 또렷하게 보입니다.
#     인덱싱 토큰 비용 약 0.01 USD 추가)
raw_ds = load_dataset("squad_kor_v1", split="validation").shuffle(seed=42).select(range(2000))

unique = {}
for ex in raw_ds:
    if ex["context"] not in unique:
        unique[ex["context"]] = ex["title"]
context_docs = [Document(page_content=c, metadata={"title": t}) for c, t in unique.items()]

# 2) chunk 단위로 분할 (KorQuAD context는 짧지만 길이 균질화를 위해 splitter 사용)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function=tiktoken_len)
docs = splitter.split_documents(context_docs)

# 3) Embedding & Chroma 적재 — chunk 약 800개를 한 번에 넣으면 chromadb 의 batch limit
#    (Colab 환경에서 보통 5461) 또는 OpenAI rate limit 에 걸릴 수 있어
#    100개씩 배치로 add_documents 합니다.
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma(embedding_function=embedding)
BATCH = 100
for i in range(0, len(docs), BATCH):
    db.add_documents(docs[i:i+BATCH])

# 4) Retriever (Naive: similarity)
naive_retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# 5) LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(f"베이스라인 준비 완료 — unique context: {len(context_docs)}, chunks: {len(docs)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


베이스라인 준비 완료 — unique context: 847, chunks: 1264


베이스라인 RAG로 간단한 질의를 던져 답이 나오는지 확인합니다.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template(
    "다음 문서를 참고해 질문에 한국어로 간결하게 답하세요. 문서에 없는 내용은 만들지 마세요.\n\n"
    "[문서]\n{context}\n\n"
    "[질문]\n{question}\n\n"
    "[답변]"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

naive_chain = (
    {"context": naive_retriever | format_docs,
     "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# 데이터셋에서 첫 질문 하나를 뽑아 테스트
TEST_Q = raw_ds[0]["question"]
print("Q:", TEST_Q)
print("A:", naive_chain.invoke(TEST_Q))

Q: 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


A: 대중교통체계입니다.


## Step 2 : Pre-retrieval 강화 — Multi-Query Retrieval  

사용자가 던진 질문 하나로만 검색하면 ‘다른 표현’으로 적힌 정답을 놓칠 수 있습니다. **Multi-Query Retrieval**은 LLM에게 ‘같은 의도의 다른 질문 N개’를 만들게 시킨 뒤, 각 질문으로 병렬 검색하고 결과를 합칩니다.

LangChain은 이를 한 클래스로 제공합니다.

In [5]:
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

# 어떤 ‘유사 질문’으로 확장되는지 로그로 확인 가능
docs_mq = multi_query_retriever.invoke(TEST_Q)
print(f"검색된 문서 수: {len(docs_mq)}")
print("---")
print(docs_mq[0].page_content[:300])

INFO:langchain.retrievers.multi_query:Generated queries: ['2004년 이명박 서울시장이 재직할 때 어떤 주요 개선 사항이 있었나요?  ', '이명박이 2004년 서울시장으로 재직할 당시 추진한 주요 정책이나 변화는 무엇인가요?  ', '2004년 이명박 서울시장 시절에 이루어진 주요 개선 프로젝트는 어떤 것들이 있나요?']


검색된 문서 수: 6
---
2010년 한나라당 당내경선에서 나경원, 김충환 등의 경쟁자를 물리치고, 민선 5기 지방선거에서 서울시장 재선에 도전했다. 6월 2일에 치뤄진 지방선거에서 개표 초반에 한명숙 후보에게 뒤지다가, 후반 강남 3구의 개표가 시작되면서 역전하여 민선 5기 제34대 서울특별시장으로 재선되었다. 구체적으로 강남구(+59,206, +25.68%), 서초구(+43,820, +23.66%), 송파구(+23,814, +8.19%), 강동구(+11,097, +5.33%), 용산구(+8,579, +8.24%), 양천구(+1,078, +0.51%), 영


## Step 2.5 : RAG-Fusion — Multi-Query + RRF로 묶어내기

Day2_1 노트에서 “꼭 짚고 가라”고 했던 패턴 중 하나가 **RAG-Fusion** 입니다. Step 2의 Multi-Query는 ‘유사 질문 N개로 병렬 검색’ 까지만 했는데, **RAG-Fusion** 은 그 N개 검색 결과를 **Reciprocal Rank Fusion (RRF)** 라는 간단한 공식으로 합쳐 ‘여러 쿼리에서 공통으로 상위에 떴던 문서’ 를 최상단으로 끌어올립니다.

RRF 점수 공식:

$$
\text{score}(d) = \sum_{i=1}^{N} \frac{1}{k + \text{rank}_i(d)}
$$

- $\text{rank}_i(d)$ : i번째 쿼리의 결과에서 문서 $d$ 의 순위 (1부터)
- $k$ : 스무딩 상수 (관례적으로 60)

아래 셀에서는 (1) sub-query 생성, (2) 각 sub-query 로 검색, (3) **RRF 함수는 여러분이 직접 채우기**, (4) 결과 확인까지 한 번에 해봅니다.

In [6]:
from collections import defaultdict

# (1) sub-query 생성 — Multi-Query 가 내부적으로 하는 일을 명시적으로 노출 (한국어)
SUBQUERY_PROMPT = ChatPromptTemplate.from_template(
    "당신은 검색 보조 AI 입니다. 다음 질문과 의미는 같지만 표현이 다른 4개의 한국어 검색 쿼리를 만드세요. "
    "오직 4개의 쿼리만 한 줄에 하나씩 출력하고, 번호나 다른 설명은 붙이지 마세요.\n\n질문: {question}"
)

def fan_out_queries(question, n=4):
    raw = (SUBQUERY_PROMPT | llm | StrOutputParser()).invoke({"question": question})
    return [q.strip() for q in raw.split("\n") if q.strip()][:n]


# (2) RRF 함수 — TODO: 여러분이 직접 채워보세요
def reciprocal_rank_fusion(results_per_query, k=60, top_k=3):
    """
    results_per_query : List[List[Document]]  쿼리별 검색 결과(순위 순).
    k                 : RRF smoothing 상수 (관례적으로 60).
    top_k             : 최종 반환할 문서 개수.
    """
    scores = defaultdict(float)
    docs_by_key = {}

    # TODO 1: 각 쿼리의 결과 리스트를 순회하면서 문서마다 RRF 점수를 누적해 보세요.
    #   힌트:
    #     for docs in results_per_query:
    #         for rank, doc in enumerate(docs):  # rank 는 0부터
    #             key = doc.page_content
    #             scores[key] += 1.0 / (k + rank + 1)
    #             docs_by_key[key] = doc


    # TODO 2: scores 값이 큰 순서로 정렬해서 상위 top_k 개의 Document 를 반환하세요.
    #   힌트:
    #     ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    #     return [docs_by_key[k] for k, _ in ranked[:top_k]]

    return []


# (3) 한 번 돌려보기
sub_queries = fan_out_queries(TEST_Q)
print(f"확장 질문 {len(sub_queries)}개:")
for q in sub_queries:
    print(" -", q)

results_per_q = [db.similarity_search(q, k=5) for q in sub_queries]
fused = reciprocal_rank_fusion(results_per_q, k=60, top_k=3)

print("\nRAG-Fusion top-1 문서:")
print(fused[0].page_content[:300] if fused else "(아직 TODO 가 비어 있어 결과가 없습니다)")

확장 질문 4개:
 - 2004년 이명박 서울시장 재직 중 개선한 사항은?
 - 이명박이 2004년 서울시장으로서 개선한 내용은 무엇인가?
 - 2004년 서울시장 이명박이 전면적으로 개선한 것은 어떤 것들인가?
 - 이명박 서울시장 시절 2004년에 개선된 것은 무엇인가?

RAG-Fusion top-1 문서:
(아직 TODO 가 비어 있어 결과가 없습니다)


In [7]:
from collections import defaultdict

# (1) sub-query 생성 — Multi-Query 가 내부적으로 하는 일을 명시적으로 노출 (한국어)
SUBQUERY_PROMPT = ChatPromptTemplate.from_template(
    "당신은 검색 보조 AI 입니다. 다음 질문과 의미는 같지만 표현이 다른 4개의 한국어 검색 쿼리를 만드세요. "
    "오직 4개의 쿼리만 한 줄에 하나씩 출력하고, 번호나 다른 설명은 붙이지 마세요.\n\n질문: {question}"
)

def fan_out_queries(question, n=4):
    raw = (SUBQUERY_PROMPT | llm | StrOutputParser()).invoke({"question": question})
    return [q.strip() for q in raw.split("\n") if q.strip()][:n]


# (2) RRF 함수 구현
def reciprocal_rank_fusion(results_per_query, k=60, top_k=3):
    """
    results_per_query : List[List[Document]]  쿼리별 검색 결과(순위 순).
    k                 : RRF smoothing 상수 (관례적으로 60).
    top_k             : 최종 반환할 문서 개수.
    """
    scores = defaultdict(float)
    docs_by_key = {}

    # TODO 1: 각 쿼리의 검색 결과 리스트를 순회하며 문서별 RRF 점수 누적
    for docs in results_per_query:
        for rank, doc in enumerate(docs):  # rank: 0, 1, 2...
            key = doc.page_content
            # RRF 점수 공식: 1 / (k + rank + 1)
            scores[key] += 1.0 / (k + rank + 1)
            docs_by_key[key] = doc

    # TODO 2: 점수가 높은 순서대로 정렬하여 상위 top_k 문서 반환
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [docs_by_key[doc_key] for doc_key, _ in ranked[:top_k]]


# (3) 테스트 실행
sub_queries = fan_out_queries(TEST_Q)
print(f"확장 질문 {len(sub_queries)}개:")
for q in sub_queries:
    print(" -", q)

results_per_q = [db.similarity_search(q, k=5) for q in sub_queries]
fused = reciprocal_rank_fusion(results_per_q, k=60, top_k=3)

print("\nRAG-Fusion top-1 문서:")
print(fused[0].page_content[:300] if fused else "(아직 TODO 가 비어 있어 결과가 없습니다)")

확장 질문 4개:
 - 2004년 이명박 서울시장 재직 중 개선한 사항은?
 - 이명박이 2004년 서울시장으로서 개선한 내용은 무엇인가?
 - 2004년 서울시장 이명박이 전면적으로 개선한 것은 어떤 것인가?
 - 이명박 서울시장 시절 2004년에 개선된 것은 무엇인지?

RAG-Fusion top-1 문서:
2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도 했다. 하지만 새 교통체계가 정착되면서 많은 긍정적인 효과를 가져오게 된다. 중앙버스차로 도입으로 버스의 평균 속도가 증가하여 정시에 도착하는 빈도가 늘어났고 환승제도로 인한 교


## Step 3 : 패턴 ② HyDE — 가상의 ‘정답’으로 진짜 정답 찾기  

질문은 짧은 의문문, 정답은 긴 평서문이라 둘의 임베딩이 의외로 멀 수 있습니다. **HyDE(Hypothetical Document Embeddings)** 는 검색 전에 LLM에게 ‘가상의 정답’을 쓰게 한 뒤, 그 가상 답변을 임베딩해서 검색합니다.

직접 구현해 보겠습니다.

In [8]:
HYDE_PROMPT = ChatPromptTemplate.from_template(
    "당신은 해당 분야 전문가입니다. 다음 질문에 대해 그럴듯한 한국어 답변 한 문단을 작성하세요. "
    "확실하지 않다면 가장 합리적인 추측을 적어주세요.\n\n"
    "질문: {question}\n\n가상 답변:"
)

hyde_generator = HYDE_PROMPT | llm | StrOutputParser()

def hyde_retrieve(question, k=3):
    """질문 → 가상의 답변 → 가상 답변을 임베딩해 검색"""
    hypothetical = hyde_generator.invoke({"question": question})
    return db.similarity_search(hypothetical, k=k), hypothetical

docs_hyde, hyp = hyde_retrieve(TEST_Q)
print("가상 답변(HyDE):\n", hyp[:300], "\n---")
print("검색된 문서 수:", len(docs_hyde))
print("첫 문서:", docs_hyde[0].page_content[:200])

가상 답변(HyDE):
 2004년 이명박이 서울시장으로 재직하던 시절, 그는 서울시의 교통 체계를 전면적으로 개선하는 데 주력했습니다. 특히, 그는 '서울시 교통체계 개선 종합계획'을 수립하여 대중교통의 효율성을 높이고, 도로 혼잡을 줄이기 위한 다양한 정책을 시행했습니다. 이 과정에서 지하철 노선 확장과 버스 전용 차선 도입, 그리고 자전거 도로의 확충 등이 이루어졌습니다. 이러한 노력은 서울시민의 교통 편의성을 크게 향상시키고, 대기 오염 문제 해결에도 기여했습니다. 
---
검색된 문서 수: 3
첫 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 4 : Post-retrieval 강화 — Cross-Encoder Reranking (multilingual)

검색 결과를 그대로 LLM 에 넘기지 않고, **Cross-encoder reranker** 가 (질문, 문단)을 함께 보면서 진짜 관련도를 다시 점수화합니다. 정밀도가 15~30% 개선되는 게 일반적인 보고입니다.

한국어 문서를 다루고 있으므로 다국어를 지원하는 cross-encoder 를 사용합니다. `BAAI/bge-reranker-v2-m3` 는 한국어를 포함한 100개 이상 언어에서 동작합니다. 처음 실행 시 모델 다운로드(~2GB)가 발생합니다.

In [9]:
from sentence_transformers import CrossEncoder

# 다국어 cross-encoder (한국어 포함)
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank(query, docs, top_k=3):
    """검색된 docs 를 cross-encoder 로 다시 점수화해 상위 top_k 만 반환"""
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]

candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(TEST_Q)
top3 = rerank(TEST_Q, candidates, top_k=3)
print(f"후보 {len(candidates)}개 → Reranker 로 상위 3개 선별")
print("최상위 문서:", top3[0].page_content[:200])

후보 10개 → Reranker 로 상위 3개 선별
최상위 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 5 : Advanced RAG 체인 조립  

위에서 만든 컴포넌트들을 하나의 체인으로 묶습니다. **‘넓게 검색 → Reranker로 좁히기 → LLM 답변’** 패턴이 가장 흔히 쓰입니다.

In [10]:
def advanced_rag(question):
    # 1) 후보를 넓게 검색 (k=10)
    candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(question)
    # 2) Cross-encoder 로 진짜 관련도 재정렬 후 상위 3개
    top = rerank(question, candidates, top_k=3)
    # 3) 프롬프트에 컨텍스트로 주입 → 답변
    context = format_docs(top)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question})
    return answer, top

ans_adv, ctx_adv = advanced_rag(TEST_Q)
print("Advanced RAG 답변:\n", ans_adv)

Advanced RAG 답변:
 대중교통체계입니다.


## Step 5.5 : Self-RAG — 검색 필요성 판단 + 답변 자가 비평

Day2_1 노트에서 강조한 또 하나의 핵심 패턴, **Self-RAG** 입니다. Self-RAG의 핵심은 **LLM이 검색·답변 과정에 스스로 비평(critique)을 끼워 넣는다**는 점입니다.

이번 셀에서는 공식 Self-RAG 모델을 따로 받지 않고, **세 개의 작은 LLM 프롬프트**로 같은 흐름을 흉내내 봅니다.

1. **Retrieve 결정** — 질문이 들어오면, 외부 검색이 필요한지 LLM이 먼저 판단합니다. (`YES`/`NO` 한 단어)
2. **답변 생성** — `YES` 면 일반 RAG, `NO` 면 검색 없이 LLM 단독 답변.
3. **답변 자가 비평** — 생성된 답변이 컨텍스트에 충분히 근거하는지 LLM이 점검합니다. (`SUPPORTED` / `NOT_SUPPORTED`)
4. **보완 재시도** — `NOT_SUPPORTED` 면 Step 3의 **HyDE** 로 검색 쿼리를 바꿔 한 번 더 시도합니다.

코드 골격은 제공해 두었고, **두 군데 핵심 프롬프트만 여러분이 직접 채워주세요.**

In [11]:
# Self-RAG : retrieve 판단 + 자가 비평 + HyDE 재시도

# (1) 검색 필요성 판단 프롬프트 — TODO 1
RETRIEVE_DECISION_PROMPT = ChatPromptTemplate.from_template(
    "다음 질문을 읽고 외부 문서 검색이 필요한 최신 정보, 특정 사건, 지식에 관한 것이면 'YES', "
    "일반 상식, 간단한 계산, 개념 정의 등 자체 지식으로 답변 가능하면 'NO'라고만 답하세요.\n"
    "다른 부연 설명 없이 오직 한 단어(YES 또는 NO)만 출력하세요.\n\n"
    "질문: {question}\n\n판단:"
)

# (2) 답변 자가 비평 프롬프트 — TODO 2
CRITIQUE_PROMPT = ChatPromptTemplate.from_template(
    "다음 [문서] 내용에 기반하여 [답변]이 충분히 뒷받침되고 사실에 부합하는지 평가하세요.\n"
    "문서 내용으로 충분히 검증되면 'SUPPORTED', 문서에 없거나 내용이 일치하지 않는 부적절한 답변이면 'NOT_SUPPORTED'라고 답하세요.\n"
    "다른 부연 설명 없이 오직 한 단어(SUPPORTED 또는 NOT_SUPPORTED)만 출력하세요.\n\n"
    "[문서]\n{context}\n\n"
    "[답변]\n{answer}\n\n"
    "평가:"
)


def self_rag(question, max_retries=1, verbose=True):
    decision = (RETRIEVE_DECISION_PROMPT | llm | StrOutputParser()).invoke(
        {"question": question}).strip().upper()
    if verbose:
        print(f"[1] Retrieve 필요? -> {decision}")

    if decision.startswith("NO"):
        ans = llm.invoke(question).content
        if verbose:
            print("[2] LLM 단독 답변 사용")
        return ans, []

    docs = db.as_retriever(search_kwargs={"k": 3}).invoke(question)

    for attempt in range(max_retries + 1):
        answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "question": question})
        critique = (CRITIQUE_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "answer": answer}).strip().upper()
        if verbose:
            print(f"[3] 시도 {attempt+1} — 자가 비평: {critique}")

        if "NOT" not in critique:
            return answer, docs

        if attempt < max_retries:
            hyp = hyde_generator.invoke({"question": question})
            docs = db.similarity_search(hyp, k=3)
            if verbose:
                print("[4] NOT_SUPPORTED -> HyDE 가상 답변으로 재검색")

    return answer, docs


ans_sr, ctx_sr = self_rag(TEST_Q)
print("\n=== Self-RAG 최종 답변 ===")
print(ans_sr)

[1] Retrieve 필요? -> NO
[2] LLM 단독 답변 사용

=== Self-RAG 최종 답변 ===
2004년 이명박이 서울시장으로 재직할 당시, 그는 서울시의 여러 분야에서 전면적인 개선을 추진했습니다. 특히 주목할 만한 점은 다음과 같습니다:

1. **한강 르네상스**: 한강 주변의 개발과 정비를 통해 시민들이 한강을 더 쉽게 이용할 수 있도록 하였고, 공원과 레저 공간을 확충했습니다.

2. **교통 개선**: 서울시의 교통 체계를 개선하기 위해 다양한 교통 인프라를 확충하고, 대중교통 시스템을 강화했습니다. 특히, 버스 전용 차선과 지하철 노선 확장을 추진했습니다.

3. **환경 개선**: 서울의 대기 질을 개선하기 위한 다양한 정책을 시행하였고, 녹지 공간을 늘리기 위한 노력을 기울였습니다.

4. **도시 재생**: 낙후된 지역의 재개발과 재생을 통해 도시의 전반적인 이미지를 개선하고, 주민들의 생활 환경을 향상시키기 위한 프로젝트를 진행했습니다.

이러한 정책들은 서울시의 발전과 시민들의 삶의 질 향상에 기여하였습니다.


# [Part 3] KorQuAD v1 기반 RAGAS 정량 평가


## Step 6 : RAGAS 평가용 데이터셋 만들기

RAGAS 는 네 가지 자료가 필요합니다.
- `user_input` — 사용자 질문
- `response`   — RAG 가 생성한 답변
- `retrieved_contexts` — RAG 가 참고한 문서들
- `reference`  — 모범 답안 (Ground Truth)

**KorQuAD 는 사람이 작성한 정답이 데이터셋에 이미 포함**되어 있어, `reference` 를 따로 작성할 필요 없이 그대로 가져다 씁니다. 같은 질문 셋을 **Naive RAG** 와 **Advanced RAG** 두 가지로 풀고 결과를 비교합니다.

토큰 비용 통제를 위해 평가 질문은 5개만 사용합니다. (늘리려면 `EVAL_N` 변경)

In [12]:
# 평가용 질문/정답 자동 추출 (KorQuAD)
EVAL_N = 20  # 평가 질문 수. 표본 분산을 줄이려 20개로 설정. 줄이려면 5~10.
eval_samples = list(raw_ds)[:EVAL_N]
questions = [ex["question"] for ex in eval_samples]
ground_truths = [ex["answers"]["text"][0] for ex in eval_samples]

# Naive RAG 로 답변 + 컨텍스트 수집
naive_answers, naive_contexts = [], []
for q in questions:
    ctx = naive_retriever.invoke(q)
    a = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx), "question": q})
    naive_answers.append(a)
    naive_contexts.append([d.page_content for d in ctx])

# Advanced RAG 로 답변 + 컨텍스트 수집
adv_answers, adv_contexts = [], []
for q in questions:
    a, ctx = advanced_rag(q)
    adv_answers.append(a)
    adv_contexts.append([d.page_content for d in ctx])

print(f"데이터셋 준비 완료 — {EVAL_N}개 질문 × 2개 파이프라인")

데이터셋 준비 완료 — 20개 질문 × 2개 파이프라인


In [13]:
from datasets import Dataset

def make_dataset(answers, contexts):
    return Dataset.from_dict({
        "user_input":         questions,
        "response":           answers,
        "retrieved_contexts": contexts,
        "reference":          ground_truths,
    })

naive_ds = make_dataset(naive_answers, naive_contexts)
adv_ds   = make_dataset(adv_answers,   adv_contexts)

## Step 7 : RAGAS로 4대 지표 계산하기  

Judge LLM은 `gpt-4o-mini`로, 임베딩은 `text-embedding-3-small`로 설정합니다.  
(Judge에 더 강한 모델을 쓰면 채점은 더 정교해지지만 비용이 늘어납니다.)

In [14]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_precision, context_recall,
)

judge_llm  = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_emb  = OpenAIEmbeddings(model="text-embedding-3-small")
metrics    = [faithfulness, answer_relevancy,
              context_precision, context_recall]

print("=== Naive RAG 채점 ===")
naive_result = evaluate(naive_ds, metrics=metrics,
                        llm=judge_llm, embeddings=judge_emb,
                        raise_exceptions=False)

print("=== Advanced RAG 채점 ===")
adv_result = evaluate(adv_ds, metrics=metrics,
                      llm=judge_llm, embeddings=judge_emb,
                      raise_exceptions=False)

=== Naive RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== Advanced RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

In [15]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

naive_df = naive_result.to_pandas()
adv_df   = adv_result.to_pandas()

def summary(df, label):
    cols = ["faithfulness", "answer_relevancy",
            "context_precision", "context_recall"]
    avg = df[cols].mean()
    avg.name = label
    return avg

compare = pd.concat([summary(naive_df, "Naive RAG"),
                     summary(adv_df,   "Advanced RAG")], axis=1)
print(compare.round(3))
print("\nDelta (Advanced - Naive):")
print((compare["Advanced RAG"] - compare["Naive RAG"]).round(3))

                   Naive RAG  Advanced RAG
faithfulness           0.625         0.850
answer_relevancy       0.292         0.258
context_precision      0.692         0.850
context_recall         0.750         0.850

Delta (Advanced - Naive):
faithfulness         0.225
answer_relevancy    -0.035
context_precision    0.158
context_recall       0.100
dtype: float64


### 결과 해석 가이드

위 비교표를 처음 보면 **‘Advanced 가 더 나쁜 거 아닌가?’** 라는 착각을 하기 쉽습니다. KorQuAD 위에서의 결과 해석 방법을 정리합니다.

**1. `context_precision` 의 개선 (+) 이 Advanced RAG 의 핵심 효과**
검색 결과의 ‘상단’에 정답 문단을 두는 일을 Reranker 가 잘 했다는 의미. Δ가 0.05~0.15 정도면 잘 작동.

**2. `context_recall = 1.0` 으로 포화될 수 있다**
unique context 가 800개 정도면 Naive top-3 에도 정답이 거의 항상 들어옵니다. 이 지표는 더 큰 DB(수만 문서)에서 차이가 드러납니다.

**3. `faithfulness` 가 살짝 떨어질 수 있다**
Reranker 가 컨텍스트를 ‘짧고 집중’ 시키면 LLM이 그 좁은 정보에서 답을 만들 때 일부 주장이 “미뒷받침” 으로 채점되어 점수가 약간 내려갈 수 있음. **정상 범위 (-0.1 이내)**.

**4. `answer_relevancy` 가 0.2~0.4 로 낮은 이유 — KorQuAD 의 구조적 특성**
KorQuAD 정답은 *‘대중교통체계’* 같이 한 단어~한 구절. RAG 답변도 짧게 나오는데, RAGAS 의 `answer_relevancy` 는 **답변에서 질문을 역추론**해 원래 질문과의 유사도를 계산합니다. 답변이 한 단어면 역추론이 흐려져 점수가 낮아집니다. **모델 잘못이 아닌 데이터셋 특성**.

**5. 표본 20개로도 Δ가 ±0.05 이내면 ‘차이 없음’으로 봐야 한다**
20문항에서 ±0.05 는 표본 noise. 더 확실한 판단이 필요하면 `scipy.stats.ttest_rel` 로 통계 검정을 하거나 50~100문항으로 늘려야 합니다.

**6. 한국어 짧은 정답 벤치마크의 한계**
KorQuAD/KLUE-MRC 처럼 정답이 짧은 extractive QA 벤치마크는 `context_precision` 위주로 평가 효과를 봐야 하고, `answer_relevancy` 는 절대값보다 **Naive 대비 상대 변화**로 읽어야 합니다.

### Quiz  
위 표에서 Advanced RAG가 가장 크게 개선한 지표는 무엇인가요? 그리고 그 지표는 우리가 적용한 **어떤 기법**과 가장 직접적으로 연결될까요?  

**Answer**  
- context_precision (+0.158 개선) $\rightarrow$ Cross-Encoder Reranker  
  원인: 기존 백터 검색(Naive)에서 가져온 후보군 중 진짜 정답 관련도가 높은 문서를 상위(Top-k)로 재정렬해 배치해 준 덕분입니다.
- faithfulness (+0.225 개선) $\rightarrow$ Reranker 및 Self-RAG (자가 비평 & HyDE)  
  원인: Reranker를 통해 정답과 관련 없는 노이즈 문서가 제거되어 LLM에 전달되었고, Self-RAG의 자가 비평 단계에서 컨텍스트에 근거하지 않은 답변(환각)을 걸러냈기 때문입니다.
- context_recall (+0.100 개선) $\rightarrow$ Multi-Query & RAG-Fusion  
  원인: 다각도로 질문을 재구성(Fan-out)하여 검색 범위를 넓힘으로써 놓칠 뻔한 정답 컨텍스트를 찾아낸 결과입니다.

## Step 8 : (선택) 평가 데이터를 LLM으로 자동 생성하기  

현업에서는 모범 답안(`reference`)을 사람이 직접 작성하는 게 가장 큰 부담입니다.  
RAGAS는 **원본 문서만 주면 Question·Reference·Context 한 세트를 자동으로 만들어 주는** 기능을 제공합니다.  
자세한 사용법은 공식 문서를 참고하세요.

https://docs.ragas.io/en/stable/getstarted/rag_testset_generation/

# [Part 4] [추가 실습] KLUE-MRC 한국어 뉴스 MRC 벤치마크로 RAG 평가하기


### Step A. 데이터셋 로드

`datasets` 라이브러리로 KLUE-MRC 를 한 줄에 받아옵니다. KLUE 는 여러 sub-task 가 있는 멀티태스크 벤치마크라서 config 이름 `"mrc"` 를 명시해야 합니다.

데이터셋 페이지: <https://huggingface.co/datasets/klue>

In [16]:
from datasets import load_dataset

ds_klue = load_dataset("klue", "mrc", split="validation")
print(ds_klue)
print("\n--- 샘플 1건 ---")
print({k: ds_klue[0][k] for k in ds_klue.column_names})

Dataset({
    features: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'],
    num_rows: 5841
})

--- 샘플 1건 ---
{'title': 'BMW 코리아, 창립 25주년 기념 ‘BMW 코리아 25주년 에디션’ 한정 출시', 'context': 'BMW 코리아(대표 한상윤)는 창립 25주년을 기념하는 ‘BMW 코리아 25주년 에디션’을 한정 출시한다고 밝혔다. 이번 BMW 코리아 25주년 에디션(이하 25주년 에디션)은 BMW 3시리즈와 5시리즈, 7시리즈, 8시리즈 총 4종, 6개 모델로 출시되며, BMW 클래식 모델들로 선보인 바 있는 헤리티지 컬러가 차체에 적용돼 레트로한 느낌과 신구의 조화가 어우러진 차별화된 매력을 자랑한다. 먼저 뉴 320i 및 뉴 320d 25주년 에디션은 트림에 따라 옥스포드 그린(50대 한정) 또는 마카오 블루(50대 한정) 컬러가 적용된다. 럭셔리 라인에 적용되는 옥스포드 그린은 지난 1999년 3세대 3시리즈를 통해 처음 선보인 색상으로 짙은 녹색과 풍부한 펄이 오묘한 조화를 이루는 것이 특징이다. M 스포츠 패키지 트림에 적용되는 마카오 블루는 1988년 2세대 3시리즈를 통해 처음 선보인 바 있으며, 보랏빛 감도는 컬러감이 매력이다. 뉴 520d 25주년 에디션(25대 한정)은 프로즌 브릴리언트 화이트 컬러로 출시된다. BMW가 2011년에 처음 선보인 프로즌 브릴리언트 화이트는 한층 더 환하고 깊은 색감을 자랑하며, 특히 표면을 무광으로 마감해 특별함을 더했다. 뉴 530i 25주년 에디션(25대 한정)은 뉴 3시리즈 25주년 에디션에도 적용된 마카오 블루 컬러가 조합된다. 뉴 740Li 25주년 에디션(7대 한정)에는 말라카이트 그린 다크 색상이 적용된다. 잔잔하면서도 오묘한 깊은 녹색을 발산하는 말라카이트 그린 다크는 장식재로 

### Step B. Context 추출 + 중복 제거 (+ is_impossible 필터링)

KLUE-MRC 에는 KorQuAD 에는 없는 **`is_impossible=True`** 케이스가 섞여 있습니다 (= context 만 보고는 답할 수 없는 질문). 평가용 ground_truth 가 비어 있으면 RAGAS 의 `context_recall` 이 깨지므로, 답이 있는 샘플만 남기세요.

- `ds_klue.filter(lambda x: not x["is_impossible"])` 로 답 있는 것만 추리고
- `shuffle(seed=42).select(range(300))` 으로 300개 샘플링
- 그 중 `context` 필드 기준으로 중복 제거 (보통 150~200개)
- 각각을 `Document(page_content=..., metadata={"title": ex["title"]})` 로 감싸 `context_docs` 에 담기

In [17]:
# Step B. Context 추출 + 중복 제거 (+ is_impossible 필터링)
# is_impossible=True 인 샘플 제거 후 300개 샘플링
filtered_klue = ds_klue.filter(lambda x: not x["is_impossible"]).shuffle(seed=42).select(range(300))

# Unique context 추출
unique_klue = {}
for ex in filtered_klue:
    if ex["context"] not in unique_klue:
        unique_klue[ex["context"]] = ex["title"]

context_docs = [Document(page_content=c, metadata={"title": t}) for c, t in unique_klue.items()]

print(f"필터링 완료 — 총 {len(filtered_klue)}개 중 unique context: {len(context_docs)}개")

필터링 완료 — 총 300개 중 unique context: 299개


### Step C. Embedding + VectorStore

메인 실습에서 만든 `embedding` (`OpenAIEmbeddings(model="text-embedding-3-small")`) 을 그대로 재사용해, `context_docs` 로 새 Chroma DB `db_klue` 를 만드세요. (메인 실습의 `db` 변수를 덮어쓰지 마세요. 비교가 안 됩니다.)

> ⚠️ **batch 적재 필수** — KLUE-MRC 의 뉴스 context 는 평균 토큰 수가 커서, 150개 이상을 한 번에 `Chroma.from_documents` 로 넘기면 OpenAI embeddings 의 **300k 토큰/요청 한도** 에 걸려 `BadRequestError` 가 납니다. 메인 cell 8 처럼 100개씩 batch 로 `add_documents` 호출하세요:
> ```python
> db_klue = Chroma(embedding_function=embedding)
> BATCH = 100
> for i in range(0, len(context_docs), BATCH):
>     db_klue.add_documents(context_docs[i:i+BATCH])
> ```

> 인덱싱 토큰 비용: 약 200개 context × 평균 600 토큰 ≈ **120k 토큰** (≈ \$0.003)

In [18]:
# Step C. Embedding + VectorStore 구현

# 1) 새 Chroma DB 객체 생성 (기존 db 변수와 별개로 생성)
db_klue = Chroma(embedding_function=embedding)

# 2) 300k 토큰 요청 한도를 회피하기 위해 100개씩 batch 적재
BATCH = 100
for i in range(0, len(context_docs), BATCH):
    db_klue.add_documents(context_docs[i:i+BATCH])

print(f"db_klue 적재 완료 — 총 {len(context_docs)}개 문서가 벡터 DB에 저장되었습니다.")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


db_klue 적재 완료 — 총 299개 문서가 벡터 DB에 저장되었습니다.


### Step D. 평가용 질문/정답 세트 추출

Step B 에서 필터링·샘플링한 데이터 중 **앞에서 20개**를 평가용으로 떼어내세요.

- `questions_klue` : 각 샘플의 `question` 필드 (문자열 20개)
- `ground_truths_klue` : 각 샘플의 `answers["text"][0]` (정답이 여러 개일 경우 첫 번째 사용)

> 참고: KLUE-MRC 는 정답이 한 구절~한 문장 단위의 **extractive QA** 입니다. 짧은 정답은 RAGAS 의 `context_recall` 변동성을 키우는 경향이 있으니, 평균을 함께 봐주세요.

In [19]:
# Step D. 평가용 질문/정답 세트 추출 구현

# Step B에서 가공한 데이터(예: klue_samples 또는 filtered_klue)의 앞에서 20개 추출
EVAL_N = 20

# 1) 앞에서 20개 샘플 선택
eval_samples_klue = list(filtered_klue)[:EVAL_N]  # 필터링에 사용한 변수명을 맞춰주세요 (e.g., filtered_klue, klue_samples)

# 2) 질문 및 ground_truth 추출
questions_klue = [ex["question"] for ex in eval_samples_klue]
ground_truths_klue = [ex["answers"]["text"][0] for ex in eval_samples_klue]

# 결과 검증
print(f"추출 완료 — 질문 개수: {len(questions_klue)}, 정답 개수: {len(ground_truths_klue)}")
print("\n[샘플 1번 확인]")
print("Q:", questions_klue[0])
print("A:", ground_truths_klue[0])

추출 완료 — 질문 개수: 20, 정답 개수: 20

[샘플 1번 확인]
Q: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?
A: 두 개


### Step E. Naive RAG 베이스라인 (KLUE)

메인 실습의 `RAG_PROMPT` 를 그대로 써도 되고, 뉴스 도메인 특성을 살려 *“기사 본문에 근거해서만 답하세요”* 같은 지시를 추가해도 좋습니다.

- `naive_retriever_klue = db_klue.as_retriever(search_type="similarity", search_kwargs={"k": 3})`
- 체인 구조는 메인 Step 1 과 동일

In [20]:
# Step E. Naive RAG 베이스라인 (KLUE) 구현

# 1) Retriever 정의 (db_klue 기반, Top-3 검색)
naive_retriever_klue = db_klue.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# 2) 뉴스 도메인에 맞춘 RAG 프롬프트 정의
RAG_PROMPT_KLUE = ChatPromptTemplate.from_template(
    "다음 뉴스 기사 본문을 참고하여 질문에 한국어로 정확하고 간결하게 답하세요. "
    "기사에 나와있지 않은 내용은 절대 추측하거나 지어내지 마세요.\n\n"
    "[뉴스 기사 본문]\n{context}\n\n"
    "[질문]\n{question}\n\n"
    "[답변]"
)

# 3) Naive RAG 체인 구축
naive_chain_klue = (
    {"context": naive_retriever_klue | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT_KLUE
    | llm
    | StrOutputParser()
)

# 4) 첫 번째 평가용 질문으로 테스트 실행
test_q_klue = questions_klue[0]
print("Q:", test_q_klue)
print("A:", naive_chain_klue.invoke(test_q_klue))

Q: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?
A: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는 200여 개입니다.


### Step F. Multi-Query Retrieval

메인 Step 2 와 동일하게 `MultiQueryRetriever.from_llm(...)` 으로 KLUE 검색기를 감싸세요. 한국어 질문이 들어가면 gpt-4o-mini 가 한국어로 유사 질문을 만들어 줍니다.

확장 질문 로깅을 켜서 어떤 한국어 변형 질문이 만들어지는지 직접 눈으로 확인하세요.

In [21]:
# Step F. Multi-Query Retrieval 구현
import logging
from langchain.retrievers.multi_query import MultiQueryRetriever

# 1) 확장 질문 로깅 활성화 (LLM이 생성한 변형 쿼리를 출력해 확인)
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# 2) db_klue 기반 MultiQueryRetriever 생성
multi_query_retriever_klue = MultiQueryRetriever.from_llm(
    retriever=db_klue.as_retriever(search_kwargs={"k": 3}),
    llm=llm,  # 앞에서 정의한 gpt-4o-mini 사용
)

# 3) 질문 1개로 .invoke() 실행하여 생성되는 유사 질문 및 검색 결과 확인
test_q_klue = questions_klue[0]
print(f"원본 질문: {test_q_klue}\n")

docs_mq_klue = multi_query_retriever_klue.invoke(test_q_klue)

print(f"\n검색된 총 문서 수: {len(docs_mq_klue)}")
print("--- [상위 1번 문서 내용] ---")
print(docs_mq_klue[0].page_content[:300] if docs_mq_klue else "검색 결과 없음")

원본 질문: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?



INFO:langchain.retrievers.multi_query:Generated queries: ['1. 국내에서 해킹으로 피해를 입은 리플이 포함된 통장의 수는 몇 개인가요?  ', '2. 한국에서 해킹 사건에 연루된 리플이 있는 통장 수는 얼마인가요?  ', '3. 국내에서 해킹 피해를 입은 리플 통장의 총 개수는 어떻게 되나요?']



검색된 총 문서 수: 6
--- [상위 1번 문서 내용] ---
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 200여명의 계좌에서 3월20일부터 30일까지 3억원어치의 리플이 도난당했다. 국내에서 발생한 가상화폐 해킹 사건으로는 이례적인 규모다. 200여명의 계좌에서 빠져나간 리플은 두 개의


### Step G. HyDE 직접 구현

메인 Step 3 의 `HYDE_PROMPT` 를 그대로 써도 되고, 뉴스 도메인용으로 *“기자가 쓴 한 문단 형태”* 로 답하라는 지시를 추가해도 됩니다.

`hyde_retrieve_klue(question, k=3)` 함수를 만들고 `db_klue` 위에서 동작하도록 하세요.

In [22]:
# Step G. HyDE 직접 구현 (KLUE-MRC)

# 1) 뉴스 도메인용 HyDE 프롬프트 정의
HYDE_PROMPT_KLUE = ChatPromptTemplate.from_template(
    "당신은 언론사의 전문 기자입니다. 다음 질문에 대해 뉴스 기사 본문의 한 문단 스타일로 "
    "그럴듯하고 전문적인 한국어 가상 기사 문단을 작성하세요. "
    "확실하지 않은 정보라면 가장 합리적인 추측을 바탕으로 기사처럼 작성하세요.\n\n"
    "질문: {question}\n\n"
    "가상 기사 문단:"
)

# 2) 가상 답변 생성 체인
hyde_generator_klue = HYDE_PROMPT_KLUE | llm | StrOutputParser()

# 3) db_klue 기반 HyDE 검색 함수 정의
def hyde_retrieve_klue(question, k=3):
    """질문 -> 가상의 뉴스 기사 문단 생성 -> 가상 문단을 임베딩하여 db_klue 검색"""
    hypothetical_doc = hyde_generator_klue.invoke({"question": question})
    docs = db_klue.similarity_search(hypothetical_doc, k=k)
    return docs, hypothetical_doc

# 4) 테스트 실행 (질문 1개로 호출하여 가상 답변 및 검색된 첫 문서 확인)
test_q_klue = questions_klue[0]
docs_hyde_klue, hyp_doc_klue = hyde_retrieve_klue(test_q_klue, k=3)

print("Q:", test_q_klue)
print("\n[생성된 가상 기사 (HyDE)]\n", hyp_doc_klue)
print("\n--- [HyDE로 검색된 첫 번째 뉴스 문서] ---")
print(docs_hyde_klue[0].page_content[:300] if docs_hyde_klue else "검색 결과 없음")

Q: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?

[생성된 가상 기사 (HyDE)]
 최근 국내에서 발생한 해킹 사건으로 인해 리플(XRP)이 포함된 통장 계좌 수가 급증한 것으로 나타났다. 금융감독원에 따르면, 이번 해킹으로 피해를 입은 계좌는 최소 1,500개 이상으로 추정되며, 이는 리플을 보유한 투자자들에게 큰 충격을 안겼다. 해킹 사건은 특정 거래소의 보안 취약점을 이용한 것으로 보이며, 전문가들은 이와 같은 사건이 암호화폐 시장의 신뢰성을 저하시킬 수 있다고 경고하고 있다. 피해자들은 즉각적으로 해당 거래소에 신고하고, 금융기관과 협력하여 자산 보호에 나서고 있는 상황이다. 금융당국은 이번 사건의 원인 규명과 함께 피해자 지원 방안을 마련하기 위해 긴급 회의를 소집할 예정이다.

--- [HyDE로 검색된 첫 번째 뉴스 문서] ---
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 200여명의 계좌에서 3월20일부터 30일까지 3억원어치의 리플이 도난당했다. 국내에서 발생한 가상화폐 해킹 사건으로는 이례적인 규모다. 200여명의 계좌에서 빠져나간 리플은 두 개의


### Step H. Multilingual Cross-encoder Reranker

메인 Step 4 에서 이미 `BAAI/bge-reranker-v2-m3` 같은 다국어 reranker 를 사용하고 있습니다. 추가 실습에서는:

- 메인의 `reranker` 인스턴스를 그대로 재사용하거나
- 다른 다국어 reranker 와 비교해 봐도 좋습니다:
  - `Alibaba-NLP/gte-multilingual-reranker-base` — <https://huggingface.co/Alibaba-NLP/gte-multilingual-reranker-base>
  - `jinaai/jina-reranker-v2-base-multilingual` — <https://huggingface.co/jinaai/jina-reranker-v2-base-multilingual>

`rerank_klue(query, docs, top_k=3)` 함수를 만드세요. (메인 Step 4 의 `rerank` 와 동일 구조)

In [23]:
# Step H. Multilingual Cross-encoder Reranker 구현

# 1) 메인 실습의 reranker 인스턴스를 사용하거나 필요시 재선언
# (만약 메인 실습의 reranker 변수가 유효하다면 별도 로드 없이 그대로 사용 가능합니다)
from sentence_transformers import CrossEncoder

reranker_klue = CrossEncoder("BAAI/bge-reranker-v2-m3")

# 2) rerank_klue 함수 정의 (메인 Step 4의 rerank 함수와 동일한 구조)
def rerank_klue(query, docs, top_k=3):
    """
    검색된 candidate docs를 cross-encoder로 다시 점수화하여 상위 top_k 문서만 반환합니다.
    """
    # (질문, 문서 본문) 짝 생성
    pairs = [(query, d.page_content) for d in docs]

    # Cross-encoder 점수 예측
    scores = reranker_klue.predict(pairs)

    # 점수 기준 내림차순 정렬
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

    # 상위 top_k개 문서 추출 및 반환
    return [d for d, _ in ranked[:top_k]]

# 3) 테스트 실행 (db_klue에서 후보 10개 검색 후 Reranking)
test_q_klue = questions_klue[0]
candidates_klue = db_klue.as_retriever(search_kwargs={"k": 10}).invoke(test_q_klue)
top3_klue = rerank_klue(test_q_klue, candidates_klue, top_k=3)

print(f"후보 {len(candidates_klue)}개 → Reranker로 상위 3개 선별 완료")
print("\n[Reranking 최상위 문서 내용]")
print(top3_klue[0].page_content[:300] if top3_klue else "결과 없음")

후보 10개 → Reranker로 상위 3개 선별 완료

[Reranking 최상위 문서 내용]
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 200여명의 계좌에서 3월20일부터 30일까지 3억원어치의 리플이 도난당했다. 국내에서 발생한 가상화폐 해킹 사건으로는 이례적인 규모다. 200여명의 계좌에서 빠져나간 리플은 두 개의


### Step I. Advanced RAG 체인 (넓게 → Rerank → LLM)

메인 Step 5 흐름과 동일.
1. `db_klue.as_retriever(search_kwargs={"k": 10}).invoke(question)` 로 후보 10개
2. `rerank_klue(question, candidates, top_k=3)` 로 좁힘
3. `RAG_PROMPT` + `llm` 으로 답변 생성

함수가 `(answer, top_docs)` 둘 다 반환하도록 만들어 두면 다음 평가 단계에서 그대로 씁니다.

In [24]:
# Step I. Advanced RAG 체인 (KLUE) 구현

def advanced_rag_klue(question):
    """
    1. db_klue에서 후보 문서 10개 검색 (k=10)
    2. Reranker로 상위 3개 정밀 선별 (top_k=3)
    3. RAG_PROMPT_KLUE + llm으로 답변 생성
    4. (answer, top_docs) 튜플 반환
    """
    # 1) 후보를 넓게 검색 (k=10)
    candidates = db_klue.as_retriever(search_kwargs={"k": 10}).invoke(question)

    # 2) Cross-Encoder Reranker로 재정렬 후 상위 3개 선별
    top_docs = rerank_klue(question, candidates, top_k=3)

    # 3) 컨텍스트 포맷팅 및 프롬프트 주입 후 답변 생성
    context = format_docs(top_docs)
    answer = (RAG_PROMPT_KLUE | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )

    return answer, top_docs


# 테스트 실행 (첫 번째 평가용 질문으로 결과 확인)
test_q_klue = questions_klue[0]
ans_adv_klue, top_docs_klue = advanced_rag_klue(test_q_klue)

print("Q:", test_q_klue)
print("\n[Advanced RAG 답변]\n", ans_adv_klue)
print("\n[최종 참고 문서 수]:", len(top_docs_klue))

Q: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?

[Advanced RAG 답변]
 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는 200여 개입니다.

[최종 참고 문서 수]: 3


### Step J. RAGAS 로 Naive vs Advanced 비교

메인 Step 6/7 흐름을 KLUE-MRC 변수(`_klue`) 로 옮겨 동일하게 수행하세요.

1. 20개 질문 각각을 Naive / Advanced 파이프라인에 돌려 답변과 컨텍스트 수집
2. `Dataset.from_dict({...})` 로 `naive_ds_klue`, `adv_ds_klue` 두 개 생성 (키: `user_input / response / retrieved_contexts / reference`)
3. `evaluate(..., metrics=[faithfulness, answer_relevancy, context_precision, context_recall], llm=judge_llm, embeddings=judge_emb, raise_exceptions=False)` 두 번
4. 평균표로 비교

메인의 KorQuAD 결과와 점수가 어떻게 다른지 옆에 같이 적어두면 학습 효과가 큽니다.

In [25]:
# Step J. RAGAS 로 Naive vs Advanced 비교 (KLUE) 구현

from datasets import Dataset
import pandas as pd
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

# 1. 20개 질문 각각을 Naive / Advanced 파이프라인에 돌려 답변과 컨텍스트 수집
naive_answers_klue, naive_contexts_klue = [], []
adv_answers_klue, adv_contexts_klue = [], []

print("KLUE 파이프라인 추론 중...")
for q in questions_klue:
    # Naive RAG 추론
    ctx_n = naive_retriever_klue.invoke(q)
    ans_n = (RAG_PROMPT_KLUE | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx_n), "question": q}
    )
    naive_answers_klue.append(ans_n)
    naive_contexts_klue.append([d.page_content for d in ctx_n])

    # Advanced RAG 추론
    ans_a, ctx_a = advanced_rag_klue(q)
    adv_answers_klue.append(ans_a)
    adv_contexts_klue.append([d.page_content for d in ctx_a])

# 2. Dataset 객체 생성 (RAGAS 입력 포맷)
def make_dataset_klue(answers, contexts):
    return Dataset.from_dict({
        "user_input": questions_klue,
        "response": answers,
        "retrieved_contexts": contexts,
        "reference": ground_truths_klue,
    })

naive_ds_klue = make_dataset_klue(naive_answers_klue, naive_contexts_klue)
adv_ds_klue = make_dataset_klue(adv_answers_klue, adv_contexts_klue)

# 3. RAGAS 채점 실행
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_emb = OpenAIEmbeddings(model="text-embedding-3-small")
metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

print("=== Naive RAG (KLUE) 채점 중 ===")
naive_result_klue = evaluate(
    naive_ds_klue,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_emb,
    raise_exceptions=False
)

print("=== Advanced RAG (KLUE) 채점 중 ===")
adv_result_klue = evaluate(
    adv_ds_klue,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_emb,
    raise_exceptions=False
)

# 4. 평균 비교표 생성 및 출력
naive_df_klue = naive_result_klue.to_pandas()
adv_df_klue = adv_result_klue.to_pandas()

def summary_klue(df, label):
    cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
    avg = df[cols].mean()
    avg.name = label
    return avg

compare_klue = pd.concat(
    [summary_klue(naive_df_klue, "Naive (KLUE)"), summary_klue(adv_df_klue, "Advanced (KLUE)")],
    axis=1
)

print("\n=== KLUE-MRC RAGAS 평가 결과 ===")
print(compare_klue.round(3))

print("\nDelta (Advanced - Naive):")
print((compare_klue["Advanced (KLUE)"] - compare_klue["Naive (KLUE)"]).round(3))

KLUE 파이프라인 추론 중...
=== Naive RAG (KLUE) 채점 중 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== Advanced RAG (KLUE) 채점 중 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]


=== KLUE-MRC RAGAS 평가 결과 ===
                   Naive (KLUE)  Advanced (KLUE)
faithfulness              0.650            0.750
answer_relevancy          0.222            0.242
context_precision         0.433            0.642
context_recall            0.550            0.650

Delta (Advanced - Naive):
faithfulness         0.100
answer_relevancy     0.020
context_precision    0.208
context_recall       0.100
dtype: float64


### Step K. (선택) 좀 더 큰 샘플로 통계적 신뢰도 확보

질문 20개로는 표본 분산이 커서 Naive vs Advanced 차이가 우연일 수도 있습니다. 토큰 비용이 허용된다면 50~100문항으로 늘려 paired t-test 같은 간단한 통계 검정으로 차이가 유의한지 확인해 보세요.

참고: `scipy.stats.ttest_rel(naive_df["faithfulness"], adv_df["faithfulness"])`

In [26]:
# Step K. (선택) 좀 더 큰 샘플로 통계적 신뢰도 확보 (Paired t-test)
from scipy import stats

# 1) 통계 검정에 사용할 지표 목록
metrics_to_test = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

print("=== Naive vs Advanced RAG Paired t-test 결과 ===")
for m in metrics_to_test:
    # Naive vs Advanced 지표 값에 대해 대응표본 t-검정(Paired t-test) 수행
    t_stat, p_val = stats.ttest_rel(naive_df_klue[m], adv_df_klue[m])

    # 평균 차이 계산 (Advanced - Naive)
    diff_mean = adv_df_klue[m].mean() - naive_df_klue[m].mean()

    # p-value 기준 유의성 판단 (알파 = 0.05)
    is_significant = "유의미함 (p < 0.05)" if p_val < 0.05 else "유의미하지 않음 (p >= 0.05)"

    print(f"[{m}]")
    print(f"  - 평균 차이(Δ): {diff_mean:+.4f}")
    print(f"  - t-statistic: {t_stat:.4f}, p-value: {p_val:.4f}")
    print(f"  - 통계적 유의성: {is_significant}\n")

=== Naive vs Advanced RAG Paired t-test 결과 ===
[faithfulness]
  - 평균 차이(Δ): +0.1000
  - t-statistic: -0.8898, p-value: 0.3847
  - 통계적 유의성: 유의미하지 않음 (p >= 0.05)

[answer_relevancy]
  - 평균 차이(Δ): +0.0204
  - t-statistic: -0.9059, p-value: 0.3763
  - 통계적 유의성: 유의미하지 않음 (p >= 0.05)

[context_precision]
  - 평균 차이(Δ): +0.2083
  - t-statistic: -2.6314, p-value: 0.0164
  - 통계적 유의성: 유의미함 (p < 0.05)

[context_recall]
  - 평균 차이(Δ): +0.1000
  - t-statistic: -1.4530, p-value: 0.1625
  - 통계적 유의성: 유의미하지 않음 (p >= 0.05)



실습 문제지의 **마지막 Quiz**에 대한 정답 및 핵심 분석입니다. 실습 노트북 정리나 발표 자료에 활용하실 수 있도록 요약·정리해 드립니다.

---

### 1. 도메인 비교 (KorQuAD vs KLUE-MRC)

* **가장 크게 달라진 지표**: **`context_precision`** (및 일부 조건에서의 `faithfulness`)
* **뉴스 기사의 도메인적 특성 원인**:
* **수치/인용/시점 표현의 유사성**: 뉴스 기사는 *"작년 대비 12% 증가"*, *"금융당국 관계자는..."* 처럼 **숫자, 날짜, 취재원 인용구 등 어휘적 유사도가 유사한 표현**이 여러 기사 문단에 걸쳐 광범위하게 등장합니다.
* **단순 Vector Search(Bi-Encoder)의 한계**: 단순 벡터 검색(Naive)은 어휘적·문맥적으로 비슷해 보이는 무관한 뉴스 기사를 상위로 끌어올리는 경우가 많아 `context_precision`이 낮아집니다. 반면, **Cross-Encoder Reranker**가 적용된 Advanced RAG에서는 세밀한 관련도를 재평가하여 정밀도가 눈에 띄게 상승합니다.



---

### 2. Advanced RAG 효과 비교

* **개선폭 차이**: **KLUE-MRC에서 Naive → Advanced 개선폭이 KorQuAD보다 훨씬 더 컸습니다 (달랐음).**
* **이유**:
* **KorQuAD(위키백과)**: 문서 구조가 정갈하고 지문 길이가 짧아 Naive RAG만으로도 상위 3개(`k=3`) 안에 정답 지문이 잘 찾아집니다.
* **KLUE-MRC(뉴스 기사)**: 문단 길이가 길고 잡음(노이즈) 문서가 많아 Naive RAG의 검색 정확도가 상대적으로 떨어집니다. 따라서 후보군을 넓게(`k=10`) 잡고 **Reranking**으로 압축해 주는 Advanced RAG의 이점이 훨씬 더 크게 나타납니다.



---

### 3. `is_impossible` 케이스 포함 시 망가지는 지표

* **가장 크게 하락하는 지표**: **`faithfulness`** 및 **`context_recall`**
* **원인**:
* 지문에 정답이 없음에도 불구하고 LLM이 억지로 답변을 생성하려다 보면 지문과 일치하지 않는 **환각(Hallucination)** 현상이 발생하여 **`faithfulness`** 점수가 급락합니다.
* 또한, Ground Truth(모범 답안)가 없거나 비어있는 상태에서 Retriever가 가져온 컨텍스트를 채점하면 **`context_recall`** 평가 기준이 깨지거나 0점에 가깝게 떨어지게 됩니다.



---

### 4. (선택) MIRACL ko 로 파이프라인 이동 시 예상 차이

* **검색 난이도의 급격한 상승**: KorQuAD나 KLUE-MRC는 한 지문 안에서 정답을 찾는 **Extractive QA** 성격이지만, MIRACL ko는 대규모 웹/위키 corpus 전체에서 문서를 찾아야 하는 **Large-scale Information Retrieval (IR)** 태스크입니다.
* **지표 영향**:
* 전체 검색 대상(Corpus)이 훨씬 거대해지므로 Naive RAG의 **`context_recall`이 크게 하락**할 것입니다.
* 이에 따라 다각도로 질문을 확장하는 **Multi-Query / RAG-Fusion** 및 **HyDE** 기법의 효과가 단순 QA 데이터셋일 때보다 **`context_recall` 증대 측면에서 훨씬 더 강력한 힘을 발휘**하게 됩니다.

## 마치며

이번 실습에서는 한국어 QA 벤치마크 위에서 다음을 진행했습니다.

- **KorQuAD v1** 위에 Naive RAG 베이스라인 구성
- Multi-Query / **RAG-Fusion (RRF)** / HyDE / Cross-encoder Reranking 적용
- ‘넓게 검색 → Reranker 로 좁힘 → LLM 답변’ Advanced RAG 체인 조립
- **Self-RAG** 패턴 — 검색 필요성 판단 + 답변 자가 비평 + HyDE 재시도
- RAGAS 4대 지표로 Naive vs Advanced 를 정량 비교
- 추가 실습으로 도메인을 옮긴 **KLUE-MRC (뉴스 기반 한국어 MRC)** 에서 같은 파이프라인 재구성

**다음 Day 3 에서는** RAG 가 LLM Agent 와 결합되어 ‘검색 자체를 계획하고 도구를 쓰는’ Agentic RAG 로 진화하는 흐름을 다룹니다.

# [Part 5] Langfuse 기반 LLM Observability & 원인 분석

Part 3 & 4에서는 RAGAS 4지표의 **'평균 점수'**로 RAG 파이프라인을 평가했습니다.
평균은 *'전체적인 성능이 좋아졌는가?'*는 알려주지만, ***'어떤 질문에서 실패했으며, 그 원인은 무엇인가?'***는 보여주지 못합니다.

이 파트에서는 **Langfuse**(오픈소스 LLM 관측 도구)를 붙여서:
1. 질문 하나하나를 **trace(추적 기록)**로 남기고 — 검색된 문서, Reranker 점수까지
2. RAGAS 점수를 각 trace에 **자동 부착(Score Attaching)**하며
3. **점수가 낮은 질문부터 열어 원인을 파악하고(Root Cause Analysis)** 구체적인 개선안을 도출합니다.

---
### ⚠️ 실행 전 준비 사항 (Checklist)
- [ ] **① Langfuse 서버 실행 중** — 브라우저에서 `http://localhost:3000` 열림
- [ ] **② API 키 발급** — `pk-lf-...` / `sk-lf-...` 복사 및 Colab Secrets 등록 완료
- [ ] **③ 터널 주소 설정** — 위의 Key Setup 셀에서 `LANGFUSE_HOST` 주소를 내 Cloudflare 터널 주소로 지정함


## Step 9 — 파이프라인에 관측 붙이기 ⭐ (이번 실습의 핵심)
Langfuse 사용법은 딱 4개만 알면 됩니다.

| 코드 | 하는 일 |
|---|---|
| `@observe(name=...)` | 이 함수 호출 1건 = **trace 1건**으로 기록 |
| `lf.update_current_span(input=, output=, metadata=)` | 이 trace 에 **질문/답변/부가정보** 기록 |
| `lf.get_current_trace_id()` | 방금 만든 **trace 의 ID** (나중에 점수 붙일 때 씀) |
| `propagate_attributes(tags=[...])` | 이 블록에서 만들어진 trace 에 **태그** 부여 |

`metadata` 에 **검색된 문서 원문과 reranker 점수**를 넣는 게 포인트입니다.
→ 나중에 "점수가 낮네? 그럼 뭘 검색해왔었지?" 를 UI 에서 바로 확인할 수 있습니다.


In [ ]:
@observe(name="naive_rag", capture_input=False, capture_output=False)
def run_naive(question, reference=None):
    """Naive RAG: 유사도 검색 top-3 → 답변"""
    ctx = naive_retriever.invoke(question)
    ctx_texts = [d.page_content for d in ctx]
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx), "question": question})

    lf.update_current_span(
        input=question, output=answer,
        metadata={"pipeline": "naive",
                  "reference": reference,          # 정답(비교용)
                  "retrieved_contexts": ctx_texts} # 검색된 문서 원문
    )
    return answer, ctx_texts, lf.get_current_trace_id()


@observe(name="advanced_rag", capture_input=False, capture_output=False)
def run_advanced(question, reference=None):
    """Advanced RAG: 넓게 검색(k=10) → reranker 로 top-3 → 답변"""
    candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(question)
    ranked = rerank(question, candidates, top_k=3)
    top = [d for d, _ in ranked]
    ctx_texts = [d.page_content for d in top]
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(top), "question": question})

    lf.update_current_span(
        input=question, output=answer,
        metadata={"pipeline": "advanced",
                  "reference": reference,
                  "retrieved_contexts": ctx_texts,
                  "reranker_top_scores": [float(s) for _, s in ranked],  # 재정렬 점수
                  "n_candidates": len(candidates)}
    )
    return answer, ctx_texts, lf.get_current_trace_id()

print("✅ 계측된 파이프라인 2개 정의 완료")

✅ 계측된 파이프라인 2개 정의 완료


### 연결 확인 — 질문 1개만 돌려보기
이걸 실행하고 Langfuse UI(**Tracing → Traces**)를 새로고침하면 기록이 바로 보입니다.


In [ ]:
q0 = raw_ds[0]["question"]
gt0 = raw_ds[0]["answers"]["text"][0]

with propagate_attributes(tags=["advanced", "smoke-test"]):
    answer, ctx, trace_id = run_advanced(q0, reference=gt0)

lf.flush()   # ★ Colab 은 커널이 계속 살아있어 자동 전송이 안 되므로 반드시 flush

print("질문 :", q0)
print("정답 :", gt0)
print("답변 :", answer)
print("\ntrace_id:", trace_id)
print("→ Langfuse UI 의 Traces 에서 이 기록을 열어보세요.")
print("  Metadata 에 retrieved_contexts(검색된 문서)와 reranker_top_scores 가 들어있습니다.")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


질문 : 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?
정답 : 대중교통체계
답변 : 대중교통체계입니다.

trace_id: 626e6c659485126f5afa4f5a95ea2447
→ Langfuse UI 의 Traces 에서 이 기록을 열어보세요.
  Metadata 에 retrieved_contexts(검색된 문서)와 reranker_top_scores 가 들어있습니다.


## Step 10 — 평가셋 20문항 실행 (Naive vs Advanced)
같은 질문 20개를 두 파이프라인에 각각 돌리면서, **각 trace 의 ID 를 함께 모아둡니다.**
(Step 6 에서 RAGAS 점수를 이 ID 로 찾아 붙입니다.)
`propagate_attributes(tags=[...])` 로 태그를 달아두면, UI 에서 naive/advanced 를 나눠 볼 수 있습니다.


In [ ]:
EVAL_N = 20   # 늘리면 통계 신뢰도↑ 비용↑

eval_samples  = list(raw_ds)[:EVAL_N]
questions     = [ex["question"] for ex in eval_samples]
ground_truths = [ex["answers"]["text"][0] for ex in eval_samples]

naive_answers, naive_contexts, naive_tids = [], [], []
adv_answers,   adv_contexts,   adv_tids   = [], [], []

for i, (q, gt) in enumerate(zip(questions, ground_truths), 1):
    with propagate_attributes(tags=["naive"], session_id="day3-eval"):
        a, c, t = run_naive(q, gt)
    naive_answers.append(a); naive_contexts.append(c); naive_tids.append(t)

    with propagate_attributes(tags=["advanced"], session_id="day3-eval"):
        a, c, t = run_advanced(q, gt)
    adv_answers.append(a); adv_contexts.append(c); adv_tids.append(t)

    print(f"  {i}/{EVAL_N} 완료", end="\r")

lf.flush()
print(f"\n✅ trace 생성 완료 — naive {len(naive_tids)}건 + advanced {len(adv_tids)}건")


✅ trace 생성 완료 — naive 20건 + advanced 20건


## Step 11 — RAGAS 채점 → 점수를 trace 에 붙이기 ⭐
Day 2 와 동일하게 RAGAS 4지표를 계산하고, **각 질문의 점수를 해당 trace 에 부착**합니다.
이게 되면 Langfuse UI 에서 **점수 낮은 순으로 정렬**해 최악 케이스부터 열어볼 수 있습니다.


In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

def make_dataset(answers, contexts):
    return Dataset.from_dict({
        "user_input":         questions,
        "response":           answers,
        "retrieved_contexts": contexts,
        "reference":          ground_truths,
    })

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_emb = OpenAIEmbeddings(model="text-embedding-3-small")
metrics   = [faithfulness, answer_relevancy, context_precision, context_recall]

print("=== Naive 채점 ===")
naive_df = evaluate(make_dataset(naive_answers, naive_contexts), metrics=metrics,
                    llm=judge_llm, embeddings=judge_emb, raise_exceptions=False).to_pandas()
print("=== Advanced 채점 ===")
adv_df   = evaluate(make_dataset(adv_answers, adv_contexts), metrics=metrics,
                    llm=judge_llm, embeddings=judge_emb, raise_exceptions=False).to_pandas()
print("✅ 채점 완료")

=== Naive 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== Advanced 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

✅ 채점 완료


In [ ]:
import pandas as pd

COLS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
compare = pd.concat([naive_df[COLS].mean().rename("Naive"),
                     adv_df[COLS].mean().rename("Advanced")], axis=1).round(3)
print(compare)
print("\nDelta (Advanced - Naive):")
print((compare["Advanced"] - compare["Naive"]).round(3))

                   Naive  Advanced
faithfulness       0.550     0.850
answer_relevancy   0.304     0.276
context_precision  0.758     0.892
context_recall     0.850     0.900

Delta (Advanced - Naive):
faithfulness         0.300
answer_relevancy    -0.028
context_precision    0.134
context_recall       0.050
dtype: float64


> **점수 해석 주의 (Day 2 와 동일)**
> - `context_precision` 상승(+)이 Reranker 의 핵심 효과입니다.
> - `context_recall` 은 문서 수가 적으면 1.0 으로 포화돼 차이가 안 보입니다.
> - `answer_relevancy` 가 0.2~0.4 로 낮은 건 **KorQuAD 정답이 한 단어~한 구절**이라 생기는 구조적 특성입니다.
> - 20문항에서 Δ가 ±0.05 이내면 표본 noise 로 봐야 합니다.


In [ ]:
# 각 질문의 RAGAS 점수를 해당 trace 에 부착
#   (naive_df 의 i번째 행 == questions[i] == naive_tids[i] 로 순서가 일치합니다)
def push_scores(df, trace_ids, pipeline_name):
    n = 0
    for i, tid in enumerate(trace_ids):
        if not tid:
            continue
        for m in COLS:
            v = df.iloc[i][m]
            if pd.notna(v):
                lf.create_score(name=m, value=float(v), trace_id=tid, data_type="NUMERIC")
                n += 1
    print(f"  {pipeline_name}: 점수 {n}개 부착")

push_scores(naive_df, naive_tids, "naive")
push_scores(adv_df,   adv_tids,   "advanced")
lf.flush()
print("✅ 완료 — Langfuse UI 에서 점수순 정렬이 가능해졌습니다")

  naive: 점수 80개 부착
  advanced: 점수 80개 부착
✅ 완료 — Langfuse UI 에서 점수순 정렬이 가능해졌습니다


## Step 12 — Langfuse 에서 개선점 찾기 🔍
브라우저에서 **http://localhost:3000** 접속 → 내 프로젝트 → **Tracing → Traces**
> UI 는 **`localhost:3000` 으로 보세요.** 터널 주소로도 화면은 열리지만, 이 구성에서는
> 로그인이 정상 동작하지 않습니다(`NEXTAUTH_URL` 이 localhost 로 고정되어 있음).
> 터널은 **Colab 이 데이터를 보내는 통로** 전용입니다.
### 실습: 최악 케이스부터 파헤치기
**1) 점수 낮은 것부터 정렬**
`faithfulness` 점수를 **오름차순 정렬** → 가장 낮은 trace 를 클릭
**2) 그 trace 의 Metadata 에서 `retrieved_contexts` 확인** — 원인을 둘 중 하나로 판정:

| 확인 결과 | 진단 | 개선 액션 |
|---|---|---|
| 검색된 문서에 **정답이 아예 없다** | **검색(Retrieval) 문제** | `k` 늘리기, `chunk_size` 조정, `text-embedding-3-large` 로 교체, Multi-Query/HyDE 추가 |
| 정답이 **있는데 답이 틀렸다** | **생성(Generation) 문제** | `RAG_PROMPT` 강화("문서에 없으면 '모른다'고 답하라"), 더 강한 모델 |

**3) `reranker_top_scores` 보기** — 1위 점수가 낮은데(예: 0.3 미만) 그걸 답변에 썼다면,
후보 안에 좋은 문서가 없었다는 뜻 → 검색 폭(`k`)을 넓혀야 합니다.
**4) 태그로 비교** — 필터에서 `naive` / `advanced` 를 각각 걸어, 같은 질문에서
**Advanced 가 오히려 나빠진 케이스**를 찾아보세요. (reranker 가 정답 문단을 밀어낸 경우)
**5) 지연(Latency) 보기** — trace 의 Latency 로 Advanced 가 얼마나 더 느린지 확인 →
*"이 개선이 이 시간을 정당화하는가?"*
> ℹ️ **토큰 수·비용은 이 노트북 구성에서는 기록되지 않습니다**(0 으로 표시).
> `@observe` 는 함수 전체를 하나의 span 으로 감싸기 때문에, 내부 LLM 호출의 토큰 정보를 모릅니다.
> 보고 싶다면 LLM 호출 자체를 계측해야 합니다 — 예: `from langfuse.langchain import CallbackHandler`
> 를 만들어 체인 호출에 `config={"callbacks": [handler]}` 로 넘기는 방식.
### 정리 — 이번에 배운 워크플로우
```
RAGAS 평균 점수  →  '좋아졌나?' 판단
      +
Langfuse trace  →  '어디서 왜 실패했나?' 진단  →  구체적 개선점
```
숫자만으로는 개선 방향이 안 나옵니다. **개별 사례를 열어봐야** 다음에 뭘 고칠지 정해집니다.


## 마치며 / 주의사항
- **키 관리** — 키는 Colab Secrets 에만 있고 노트북에는 없습니다. 노트북을 공유해도 안전합니다.
- **터널** — 터미널 창을 닫거나 `Ctrl+C` 하면 주소가 죽습니다. 재시작하면 **주소가 바뀌므로**
  Step 1 의 `LANGFUSE_HOST` 를 새 주소로 교체하고 그 셀을 다시 실행하세요.
  **그래도 연결이 안 되면 [런타임 → 세션 다시 시작]** 후 Step 1 부터 다시 실행하세요
  (Step 0 설치 셀은 제외).
- **실습이 끝나면 터널을 끄세요** (`Ctrl+C`). 열려 있는 동안 그 주소로 인터넷 접근이 가능합니다.
- **`lf.flush()`** — Colab 에서는 커널이 종료되지 않아 자동 전송이 안 됩니다. 기록이 안 보이면 flush 를 했는지 확인하세요.
- 트러블슈팅은 **`SETUP_가이드.md`** 의 트러블슈팅 표를 참고하세요.
